# MCP-backed memory demo

Same agent surface as `memory_skills_demo.ipynb`, but `ctx.file_backend` is an `MCPFileBackend` talking to a Python MCP server (`examples/mcp_memory_server.py`) over stdio. Use this layout when the underlying memory store is owned by a separate service (e.g. Postgres behind an MCP server) and you don't want to expose raw SQL to the agent.

Streamed via `agent.run_stream(...)` + `render_events(...)` so you can watch MCP `resources/read` and `tools/call` traffic interleave with the agent's tool calls and streamed token output.

Requires `OPENAI_API_KEY` in the environment, the `mcp` optional dependency (`pip install grasp-agents[mcp]`), and `python` on the `PATH`.

**Architectural alignment with the local-FS demo.** `MemoryProvider` is the same class — it knows the memdir layout but does no I/O itself. Reads/writes route through `ctx.file_backend`. The local demo uses `LocalFileBackend`; this one uses `MCPFileBackend(client=...)`. The agent code, memory section, and file-edit toolkit are unchanged.

The MCP server speaks two surfaces:
* **Resources** (`resources/read`) — the canonical read path. `MEMORY.md` plus each topic file is exposed as `file://<path>`.
* **Tools** (`write_file` / `stat_file` / `delete_file` / `list_dir`) — the mutating + metadata operations. `MCPFileBackend` drives these behind the scenes.

**Why one big code cell?** The MCP stdio client uses an anyio cancel scope tied to the asyncio task that opened it. Jupyter runs each cell as a fresh task, so a `connect()` in one cell + a `close()` in another raises. The whole demo lives inside one `async with MCPClient(...)` block; narrative is in the markdown cells above and below.

## Imports + setup

In [ ]:
from pathlib import Path
import shutil
import sys
import tempfile

import grasp_agents
from grasp_agents import (
    LLMAgent,
    MCPClient,
    MCPServerStdio,
    MemoryProvider,
    ProcPacketOutEvent,
    SessionContext,
    render_events,
)
from grasp_agents.file_backend import MCPFileBackend
from grasp_agents.llm_providers.openai_completions.completions_llm import (
    OpenAILLM,
    OpenAILLMSettings,
)


def make_llm() -> OpenAILLM:
    return OpenAILLM(
        model_name="openai/gpt-4o-mini",
        llm_settings=OpenAILLMSettings(temperature=0.0),
    )


async def run_and_capture(agent: LLMAgent, message: str):
    final = None
    async for event in render_events(
        agent.run_stream(message),
        show_input_messages=True,
    ):
        if isinstance(event, ProcPacketOutEvent):
            final = event.data.payloads[0]
    return final


# tempfile.mkdtemp returns the platform's "user-visible" tmp dir (on
# macOS that's ``/var/folders/...``, which symlinks to ``/private/var/
# folders/...``). The MCP server canonicalises paths via ``Path.resolve()``
# so its resource URIs come back with the resolved prefix. We call
# ``.resolve()`` on the client side too so URI lookups match.
TMPDIR = Path(tempfile.mkdtemp(prefix="mcp_memory_demo_")).resolve()
MEMDIR = TMPDIR / "memdir"
MEMDIR.mkdir()

(MEMDIR / "MEMORY.md").write_text(
    "# grasp-agents demo memory index\n\n"
    "Topics:\n\n"
    "- [user_preferences](user_preferences.md) — how the user likes their answers formatted.\n",
    encoding="utf-8",
)
# The body of `user_preferences` lives ONLY in user_preferences.md
# (served via the MCP server). MEMORY.md just links to it — the
# agent reaches the body by `Read`-ing the file (routed through
# MCP `resources/read` by the backend).
(MEMDIR / "user_preferences.md").write_text(
    "---\nname: user_preferences\ntype: user\n"
    "description: how the user likes their answers formatted\n---\n"
    "Reply in concise bullet points unless the user asks for prose.\n",
    encoding="utf-8",
)

SERVER_SCRIPT = Path(grasp_agents.__file__).parent / "examples" / "mcp_memory_server.py"
print("memdir:", MEMDIR)
print("server script:", SERVER_SCRIPT)
print("files:", sorted(p.name for p in MEMDIR.iterdir()))

## End-to-end run

Steps inside the one `async with` block:

1. Spawn the MCP server + connect → build `MCPFileBackend(client=...)`.
2. Construct `MemoryProvider(MEMDIR)` and load the snapshot via the backend (entries discovered through `list_dir` + `read_text`, both routed over MCP).
3. Run a read-shaped agent — `enable_memory=True` auto-attaches the file toolkit; the agent calls `Read` to follow the `MEMORY.md` link, and under the hood that becomes MCP `resources/read`.
4. Run the same agent shape again to **author** a topic — `Write` creates the new topic file (MCP `tools/call write_file`) and `Edit` updates `MEMORY.md` to link it.

The MCP server's two surfaces (resources + file tools) make this work end-to-end without exposing any specialized memory tools to the agent.

In [ ]:
async with MCPClient(
    "mem-server",
    server=MCPServerStdio(
        command=sys.executable,
        args=[str(SERVER_SCRIPT), str(MEMDIR)],
    ),
) as client:
    print("connected:", client.name)
    print("discovered tools:", [t.name for t in client.tools()])

    # Backend + provider — provider does no I/O itself; backend handles
    # bytes (here over MCP).
    backend = MCPFileBackend(client=client, allowed_roots=[MEMDIR])
    provider = MemoryProvider(MEMDIR)

    # ctx is constructed first so it can be passed to every agent
    # below. The SessionContext validator binds ``file_backend`` onto the
    # provider so its methods (``load`` / ``fetch_body`` / ...) don't
    # need a ctx kwarg at call time.
    ctx: SessionContext[None] = SessionContext(
        state=None, memory=provider, file_backend=backend
    )

    # --- 2. Inspect the snapshot ---
    snap = await provider.load()
    print("\nindex (first 80 chars):")
    print(snap.index[:80] if snap.index else snap.index)
    print("\nentries:")
    for e in snap.entries:
        print(f"  {e.name}  type={e.memory_type}  path={e.path}")
    print("\nfetch_body('user_preferences'):")
    print(await provider.fetch_body("user_preferences"))

    # --- 3. Agent reads MEMORY.md + the linked topic ---
    print("\n=== Step 1 — read MEMORY.md + linked topic ===")
    agent = LLMAgent[str, str, None](
        name="demo",
        ctx=ctx,
        llm=make_llm(),
        enable_memory=True,
        # tools=[] → agentic mode so enable_memory auto-attaches
        # Read / Write / Edit / Delete / Glob / Grep, here backed
        # by MCPFileBackend (calls route over MCP).
        tools=[],
        sys_prompt="You are a helpful assistant.",
        env_info=False,
        stream_llm=True,
    )
    print("wired tools:", list(agent.tools))
    final = await run_and_capture(
        agent,
        "How do I prefer my answers formatted? Look it up if you need to.",
    )
    print("\nfinal:", final)

    # --- 4. Agent authors a memory through MCP file tools ---
    print("\n=== Step 2 — author via Read/Write/Edit (MCP-backed) ===")
    agent = LLMAgent[str, str, None](
        name="demo",
        ctx=ctx,
        llm=make_llm(),
        enable_memory=True,
        tools=[],
        sys_prompt="You are a helpful assistant.",
        env_info=False,
        stream_llm=True,
    )
    final = await run_and_capture(
        agent,
        "Please remember: I'm based in Berlin (CET timezone).",
    )
    print("\nfinal:", final)

    # The provider caches the snapshot frozen-style — refresh
    # to pick up the topic file the agent just authored.
    await provider.refresh()
    snap = await provider.load()
    print("\nMCP snapshot entries now:", [e.name for e in snap.entries])

print("\nfiles on disk after the run:", sorted(p.name for p in MEMDIR.iterdir()))
print("\nMEMORY.md after the run:")
print((MEMDIR / "MEMORY.md").read_text())

## Cleanup

The MCP server subprocess was terminated when the `async with` block exited. Remove the tmpdir.

In [ ]:
shutil.rmtree(TMPDIR, ignore_errors=True)
print("removed:", TMPDIR)